# Módulo 3: Sistema de Recomendación de Destinos de Viaje
## Análisis Exploratorio de Datos (EDA)

**Universidad Nacional de Colombia**  
**Introducción a las Redes Neuronales Artificiales (IRNA)**  
**Trabajo 3 — Módulo 3**

Este notebook realiza un análisis exploratorio completo del dataset de recomendaciones de destinos de viaje,
identificando patrones relevantes para el diseño del sistema basado en Neural Collaborative Filtering (NCF).

## 1. Importación de Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import warnings
import os

warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
sns.set_palette('husl')

# Directorio de salida
OUTPUT_DIR = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Librerías cargadas exitosamente')
print(f'NumPy: {np.__version__}  |  Pandas: {pd.__version__}')

## 2. Generación del Dataset Sintético

Se genera un dataset realista que simula interacciones de 500 usuarios con 50 destinos de Colombia y Latinoamérica.
Cada usuario tiene un perfil de preferencias que sesga sus interacciones y ratings.

In [ ]:
np.random.seed(42)

# Catálogo de destinos con metadata completa
destinations_info = {
    'Cartagena':        {'category': 'Playa',      'country': 'Colombia',   'cost': 300, 'region': 'Caribe'},
    'Bogotá':           {'category': 'Ciudad',     'country': 'Colombia',   'cost': 150, 'region': 'Andina'},
    'Medellín':         {'category': 'Ciudad',     'country': 'Colombia',   'cost': 180, 'region': 'Andina'},
    'Santa Marta':      {'category': 'Playa',      'country': 'Colombia',   'cost': 250, 'region': 'Caribe'},
    'San Andrés':       {'category': 'Playa',      'country': 'Colombia',   'cost': 450, 'region': 'Caribe'},
    'Eje Cafetero':     {'category': 'Naturaleza', 'country': 'Colombia',   'cost': 200, 'region': 'Andina'},
    'Tayrona':          {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 280, 'region': 'Caribe'},
    'Leticia':          {'category': 'Aventura',   'country': 'Colombia',   'cost': 500, 'region': 'Amazonia'},
    'Salento':          {'category': 'Cultural',   'country': 'Colombia',   'cost': 120, 'region': 'Andina'},
    'Barichara':        {'category': 'Cultural',   'country': 'Colombia',   'cost': 100, 'region': 'Andina'},
    'Villa de Leyva':   {'category': 'Cultural',   'country': 'Colombia',   'cost': 130, 'region': 'Andina'},
    'Cali':             {'category': 'Ciudad',     'country': 'Colombia',   'cost': 160, 'region': 'Pacifico'},
    'Barranquilla':     {'category': 'Ciudad',     'country': 'Colombia',   'cost': 200, 'region': 'Caribe'},
    'Providencia':      {'category': 'Playa',      'country': 'Colombia',   'cost': 600, 'region': 'Caribe'},
    'Cano Cristales':   {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 400, 'region': 'Orinoquia'},
    'Bucaramanga':      {'category': 'Ciudad',     'country': 'Colombia',   'cost': 140, 'region': 'Andina'},
    'Nuqui':            {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 350, 'region': 'Pacifico'},
    'Popayan':          {'category': 'Cultural',   'country': 'Colombia',   'cost': 110, 'region': 'Andina'},
    'Manizales':        {'category': 'Naturaleza', 'country': 'Colombia',   'cost': 170, 'region': 'Andina'},
    'Pasto':            {'category': 'Cultural',   'country': 'Colombia',   'cost': 130, 'region': 'Andina'},
    'Cancun':           {'category': 'Playa',      'country': 'Mexico',     'cost': 700, 'region': 'Caribe'},
    'Ciudad de Mexico': {'category': 'Ciudad',     'country': 'Mexico',     'cost': 400, 'region': 'Centroamerica'},
    'Buenos Aires':     {'category': 'Ciudad',     'country': 'Argentina',  'cost': 500, 'region': 'Sudamerica Sur'},
    'Patagonia':        {'category': 'Aventura',   'country': 'Argentina',  'cost': 800, 'region': 'Sudamerica Sur'},
    'Cusco':            {'category': 'Cultural',   'country': 'Peru',       'cost': 450, 'region': 'Sudamerica'},
    'Machu Picchu':     {'category': 'Cultural',   'country': 'Peru',       'cost': 600, 'region': 'Sudamerica'},
    'Lima':             {'category': 'Ciudad',     'country': 'Peru',       'cost': 350, 'region': 'Sudamerica'},
    'Rio de Janeiro':   {'category': 'Playa',      'country': 'Brasil',     'cost': 650, 'region': 'Sudamerica'},
    'Pantanal':         {'category': 'Ecoturismo', 'country': 'Brasil',     'cost': 550, 'region': 'Sudamerica'},
    'Galapagos':        {'category': 'Ecoturismo', 'country': 'Ecuador',    'cost': 1200, 'region': 'Sudamerica'},
    'Quito':            {'category': 'Ciudad',     'country': 'Ecuador',    'cost': 300, 'region': 'Sudamerica'},
    'Salar de Uyuni':   {'category': 'Aventura',   'country': 'Bolivia',    'cost': 400, 'region': 'Sudamerica'},
    'La Paz':           {'category': 'Ciudad',     'country': 'Bolivia',    'cost': 250, 'region': 'Sudamerica'},
    'Montevideo':       {'category': 'Ciudad',     'country': 'Uruguay',    'cost': 450, 'region': 'Sudamerica Sur'},
    'Punta del Este':   {'category': 'Playa',      'country': 'Uruguay',    'cost': 700, 'region': 'Sudamerica Sur'},
    'Atacama':          {'category': 'Aventura',   'country': 'Chile',      'cost': 650, 'region': 'Sudamerica Sur'},
    'Santiago':         {'category': 'Ciudad',     'country': 'Chile',      'cost': 500, 'region': 'Sudamerica Sur'},
    'Isla de Pascua':   {'category': 'Cultural',   'country': 'Chile',      'cost': 1100, 'region': 'Pacifico'},
    'Roraima':          {'category': 'Aventura',   'country': 'Venezuela',  'cost': 600, 'region': 'Sudamerica'},
    'Iguazu':           {'category': 'Naturaleza', 'country': 'Argentina',  'cost': 500, 'region': 'Sudamerica Sur'},
    'Valparaiso':       {'category': 'Cultural',   'country': 'Chile',      'cost': 400, 'region': 'Sudamerica Sur'},
    'Florianopolis':    {'category': 'Playa',      'country': 'Brasil',     'cost': 550, 'region': 'Sudamerica'},
    'Amazon Ecuador':   {'category': 'Ecoturismo', 'country': 'Ecuador',    'cost': 700, 'region': 'Sudamerica'},
    'Tulum':            {'category': 'Playa',      'country': 'Mexico',     'cost': 600, 'region': 'Centroamerica'},
    'Oaxaca':           {'category': 'Cultural',   'country': 'Mexico',     'cost': 350, 'region': 'Centroamerica'},
    'San Jose CR':      {'category': 'Ciudad',     'country': 'Costa Rica', 'cost': 350, 'region': 'Centroamerica'},
    'Monteverde':       {'category': 'Ecoturismo', 'country': 'Costa Rica', 'cost': 500, 'region': 'Centroamerica'},
    'Havana':           {'category': 'Cultural',   'country': 'Cuba',       'cost': 600, 'region': 'Caribe'},
    'Cuzco Sagrado':    {'category': 'Cultural',   'country': 'Peru',       'cost': 520, 'region': 'Sudamerica'},
    'Cartagena Ind':    {'category': 'Cultural',   'country': 'Colombia',   'cost': 220, 'region': 'Caribe'},
}

destination_names = list(destinations_info.keys())
n_users = 500
n_items = len(destination_names)
category_list = list(set([d['category'] for d in destinations_info.values()]))

print(f'Destinos en catálogo: {n_items}')
print(f'Categorías: {category_list}')
print(f'Usuarios a simular: {n_users}')

In [ ]:
# Asignar perfil de preferencia a cada usuario
budget_limits = {'bajo': 300, 'medio': 600, 'alto': 9999}

user_profiles = []
for u in range(n_users):
    preferred_categories = list(np.random.choice(category_list, size=np.random.randint(1, 4), replace=False))
    budget = np.random.choice(['bajo', 'medio', 'alto'], p=[0.3, 0.5, 0.2])
    user_profiles.append({'user_id': u, 'preferred_categories': preferred_categories, 'budget': budget})

# Generar interacciones con sesgo realista
interactions = []
for user in user_profiles:
    uid = user['user_id']
    pref_cats = user['preferred_categories']
    budget_limit = budget_limits[user['budget']]

    # Distribución de cola larga en número de interacciones
    n_inter = max(3, min(40, int(np.random.exponential(15))))

    # Pesos de selección por afinidad
    weights = []
    for dname in destination_names:
        d = destinations_info[dname]
        w = 1.0
        if d['category'] in pref_cats:
            w *= 4.0
        if d['cost'] <= budget_limit:
            w *= 2.0
        weights.append(w)
    weights = np.array(weights) / np.sum(weights)

    chosen = np.random.choice(n_items, size=min(n_inter, n_items), replace=False, p=weights)

    for idx in chosen:
        dname = destination_names[idx]
        d = destinations_info[dname]
        base = 3.0 + (np.random.uniform(0.5, 2.0) if d['category'] in pref_cats else np.random.uniform(-1.0, 1.0))
        rating = float(np.clip(round(base + np.random.normal(0, 0.5), 1), 1.0, 5.0))

        interactions.append({
            'user_id': uid,
            'destination': dname,
            'item_id': idx,
            'rating': rating,
            'category': d['category'],
            'country': d['country'],
            'region': d['region'],
            'cost': d['cost'],
            'year':  np.random.choice([2022, 2023, 2024]),
            'month': np.random.randint(1, 13),
        })

df = pd.DataFrame(interactions)
print(f'Dataset generado: {df.shape[0]:,} interacciones')
df.head()

## 3. Descripción General del Dataset

In [ ]:
print('=' * 60)
print('DESCRIPCIÓN GENERAL')
print('=' * 60)
print(f'Shape:            {df.shape}')
print(f'Usuarios únicos:  {df["user_id"].nunique():,}')
print(f'Destinos únicos:  {df["destination"].nunique()}')
print(f'Países únicos:    {df["country"].nunique()}')
print(f'Categorías:       {df["category"].nunique()}')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nValores nulos: {df.isnull().sum().sum()}')

In [ ]:
print('Estadísticas descriptivas:')
display(df.describe())
print('\nPrimeras 10 filas:')
display(df.head(10))

## 4. Distribución de Ratings (Histograma + KDE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histograma ---
ax = axes[0]
ax.hist(df['rating'], bins=35, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df['rating'].mean(),   color='red',    ls='--', lw=2, label=f'Media: {df["rating"].mean():.2f}')
ax.axvline(df['rating'].median(), color='orange', ls='--', lw=2, label=f'Mediana: {df["rating"].median():.2f}')
ax.axvline(3.5, color='purple', ls=':', lw=2, label='Umbral implícito (3.5)')
ax.set_xlabel('Rating'); ax.set_ylabel('Frecuencia')
ax.set_title('Histograma de Ratings')
ax.legend()

# --- KDE ---
ax2 = axes[1]
kde = gaussian_kde(df['rating'])
xr = np.linspace(0.5, 5.5, 400)
ax2.fill_between(xr, kde(xr), alpha=0.35, color='steelblue')
ax2.plot(xr, kde(xr), color='steelblue', lw=2)
ax2.axvline(3.5, color='red', ls='--', lw=2, label='Umbral implícito (3.5)')
ax2.set_xlabel('Rating'); ax2.set_ylabel('Densidad')
ax2.set_title('Estimación de Densidad (KDE)')
ax2.legend()

plt.suptitle('Distribución de Ratings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_ratings.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Skewness: {df["rating"].skew():.3f}')
print(f'Kurtosis: {df["rating"].kurtosis():.3f}')
print(f'Ratings positivos (>3.5): {(df["rating"] > 3.5).mean()*100:.1f}%')

## 5. Top 20 Destinos Más Visitados

In [ ]:
top20 = df['destination'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top20)))
bars = ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, top20.values[::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', ha='left', fontsize=9, fontweight='bold')

ax.set_xlabel('Número de interacciones')
ax.set_title('Top 20 Destinos Más Visitados', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_top20.png'), dpi=100, bbox_inches='tight')
plt.show()

# Rating promedio por destino
dest_stats = df.groupby('destination')['rating'].agg(['mean','count']).sort_values('count', ascending=False).head(20)
dest_stats.columns = ['Rating Promedio', 'N Interacciones']
print('\nTop 20 destinos — rating y conteo:')
display(dest_stats.round(2))

## 6. Análisis de Actividad por Usuario

In [ ]:
user_activity = df.groupby('user_id').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax = axes[0]
ax.hist(user_activity, bins=30, color='coral', edgecolor='white', alpha=0.85)
ax.axvline(user_activity.mean(),   color='blue',  ls='--', lw=2, label=f'Media: {user_activity.mean():.1f}')
ax.axvline(user_activity.median(), color='green', ls='--', lw=2, label=f'Mediana: {user_activity.median():.1f}')
ax.set_xlabel('Número de interacciones'); ax.set_ylabel('Usuarios')
ax.set_title('Distribución de Interacciones por Usuario')
ax.legend()

# Curva de Lorenz
ax2 = axes[1]
sa = np.sort(user_activity)
cumfrac = np.cumsum(sa) / sa.sum()
xfrac = np.linspace(0, 1, len(sa))
ax2.plot(xfrac, cumfrac, color='steelblue', lw=2, label='Curva de Lorenz')
ax2.plot([0,1],[0,1], 'k--', alpha=0.4, label='Distribución perfecta')
ax2.fill_between(xfrac, xfrac, cumfrac, alpha=0.2, color='steelblue')
ax2.set_xlabel('Fracción acumulada de usuarios')
ax2.set_ylabel('Fracción acumulada de interacciones')
ax2.set_title('Curva de Lorenz — Actividad de Usuarios')
ax2.legend()

plt.suptitle('Análisis de Actividad por Usuario', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_usuarios.png'), dpi=100, bbox_inches='tight')
plt.show()

print(user_activity.describe())
print(f'\nUsuarios con >=3 interacciones: {(user_activity>=3).sum()} ({(user_activity>=3).mean()*100:.1f}%)')
print(f'Cold start (1 interacción):     {(user_activity==1).sum()}')

## 7. Análisis de Categorías de Destino (Pie + Barras)

In [ ]:
cat_counts = df['category'].value_counts()
cat_colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            startangle=90, colors=cat_colors)
axes[0].set_title('Distribución por Categoría (Pie Chart)', fontweight='bold')

# Barras + rating promedio en eje secundario
ax2 = axes[1]
cat_rating = df.groupby('category')['rating'].mean().reindex(cat_counts.index)
ax2.bar(cat_counts.index, cat_counts.values, color=cat_colors, alpha=0.8, edgecolor='white')
ax2.set_xticklabels(cat_counts.index, rotation=40, ha='right')
ax2.set_ylabel('Número de interacciones')
ax2.set_title('Frecuencia y Rating Promedio por Categoría', fontweight='bold')

ax3 = ax2.twinx()
ax3.plot(range(len(cat_counts)), cat_rating.values, 'ro-', lw=2, ms=7, label='Rating promedio')
ax3.set_ylabel('Rating Promedio', color='red')
ax3.tick_params(axis='y', labelcolor='red')
ax3.legend()

plt.suptitle('Análisis de Categorías de Destino', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_categorias.png'), dpi=100, bbox_inches='tight')
plt.show()

cat_summary = df.groupby('category').agg(
    N=('rating','count'), Rating_medio=('rating','mean'), Costo_medio=('cost','mean')
).round(2).sort_values('N', ascending=False)
print(cat_summary)

## 8. Distribución Geográfica por País y Región

In [ ]:
country_counts = df['country'].value_counts()
region_counts  = df['region'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras horizontales por país
ax = axes[0]
colors_c = plt.cm.tab20(np.linspace(0,1,len(country_counts)))
bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1], color=colors_c)
for bar, val in zip(bars, country_counts.values[::-1]):
    ax.text(bar.get_width()+3, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9)
ax.set_xlabel('Interacciones'); ax.set_title('Interacciones por País', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Pie de regiones
axes[1].pie(region_counts, labels=region_counts.index, autopct='%1.1f%%',
            startangle=90, colors=plt.cm.Paired(np.linspace(0,1,len(region_counts))))
axes[1].set_title('Distribución por Región', fontweight='bold')

plt.suptitle('Distribución Geográfica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_geografica.png'), dpi=100, bbox_inches='tight')
plt.show()

col_pct = df[df['country']=='Colombia'].shape[0] / len(df) * 100
print(f'Colombia representa el {col_pct:.1f}% del total de interacciones.')

## 9. Análisis Temporal: Visitas por Mes y Año

In [ ]:
month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Por mes
monthly = df.groupby('month').size()
axes[0,0].bar(range(1,13), [monthly.get(m,0) for m in range(1,13)],
              color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_xticks(range(1,13)); axes[0,0].set_xticklabels(month_names, rotation=45)
axes[0,0].set_ylabel('Interacciones'); axes[0,0].set_title('Interacciones por Mes')

# Por año
yearly = df.groupby('year').size()
bars = axes[0,1].bar(yearly.index, yearly.values, color='coral', edgecolor='white', alpha=0.85)
for bar, val in zip(bars, yearly.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontweight='bold')
axes[0,1].set_xlabel('Año'); axes[0,1].set_ylabel('Interacciones'); axes[0,1].set_title('Interacciones por Año')

# Heatmap mes-año
pivot = df.groupby(['year','month']).size().unstack(fill_value=0)
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,0],
            xticklabels=[month_names[i-1] for i in pivot.columns])
axes[1,0].set_xlabel('Mes'); axes[1,0].set_ylabel('Año'); axes[1,0].set_title('Heatmap Mes x Año')

# Rating promedio por mes
mr = df.groupby('month')['rating'].mean()
axes[1,1].plot(range(1,13), [mr.get(m,3) for m in range(1,13)], 'go-', lw=2, ms=8)
axes[1,1].fill_between(range(1,13), [mr.get(m,3) for m in range(1,13)], alpha=0.2, color='green')
axes[1,1].set_xticks(range(1,13)); axes[1,1].set_xticklabels(month_names, rotation=45)
axes[1,1].set_ylim(1,5); axes[1,1].set_ylabel('Rating Promedio'); axes[1,1].set_title('Rating Promedio por Mes')

plt.suptitle('Análisis Temporal del Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_temporal.png'), dpi=100, bbox_inches='tight')
plt.show()

## 10. Matriz de Densidad Usuario-Destino (Sparsity)

In [ ]:
n_u = df['user_id'].nunique()
n_i = df['destination'].nunique()
sparsity = 1 - len(df) / (n_u * n_i)

print(f'Usuarios: {n_u}  |  Destinos: {n_i}')
print(f'Interacciones: {len(df):,}')
print(f'Densidad: {(1-sparsity)*100:.2f}%  |  Sparsity: {sparsity*100:.2f}%')

# Submuestra 80 usuarios x 30 destinos
sample_users = sorted(df['user_id'].unique())[:80]
sample_items = df['destination'].value_counts().head(30).index.tolist()
df_sub = df[df['user_id'].isin(sample_users) & df['destination'].isin(sample_items)]
mat = df_sub.pivot_table(index='user_id', columns='destination', values='rating', aggfunc='mean')
mat = mat.reindex(index=sample_users, columns=sample_items)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(mat.fillna(0), ax=axes[0], cmap='YlOrRd',
            xticklabels=False, yticklabels=False,
            cbar_kws={'label': 'Rating (0=sin interacción)'})
axes[0].set_title(f'Matriz Usuario-Destino (submuestra 80x30)\nSparsity global: {sparsity*100:.1f}%', fontweight='bold')
axes[0].set_xlabel('Destinos (Top 30)'); axes[0].set_ylabel('Usuarios (80 primeros)')

item_counts = df['destination'].value_counts().sort_values(ascending=False)
axes[1].plot(range(len(item_counts)), item_counts.values, 'b-', lw=2)
axes[1].fill_between(range(len(item_counts)), item_counts.values, alpha=0.2)
axes[1].axhline(item_counts.mean(), color='red', ls='--', label=f'Media: {item_counts.mean():.1f}')
axes[1].set_xlabel('Destinos (por popularidad)'); axes[1].set_ylabel('Interacciones')
axes[1].set_title('Distribución Long-Tail de Popularidad', fontweight='bold')
axes[1].legend()

plt.suptitle('Análisis de Sparsity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_sparsity.png'), dpi=100, bbox_inches='tight')
plt.show()

## 11. Análisis de Correlación entre Variables Numéricas

In [ ]:
# Enriquecer con estadísticas agregadas
df['user_avg_rating']   = df.groupby('user_id')['rating'].transform('mean')
df['item_avg_rating']   = df.groupby('destination')['rating'].transform('mean')
df['user_n_inter']      = df.groupby('user_id')['rating'].transform('count')
df['item_n_inter']      = df.groupby('destination')['rating'].transform('count')

num_cols = ['rating','cost','user_avg_rating','item_avg_rating','user_n_inter','item_n_inter','month','year']
corr = df[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap correlación
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0], square=True)
axes[0].set_title('Matriz de Correlación', fontweight='bold')

# Scatter costo vs rating por categoría
dest_agg = df.groupby('destination').agg({'rating':'mean','cost':'first','category':'first'}).reset_index()
cats_u = dest_agg['category'].unique()
colors_s = plt.cm.tab10(np.linspace(0,1,len(cats_u)))
for i, cat in enumerate(cats_u):
    sub = dest_agg[dest_agg['category']==cat]
    axes[1].scatter(sub['cost'], sub['rating'], label=cat, color=colors_s[i], alpha=0.8, s=70)

z = np.polyfit(dest_agg['cost'], dest_agg['rating'], 1)
xr = np.linspace(dest_agg['cost'].min(), dest_agg['cost'].max(), 100)
axes[1].plot(xr, np.poly1d(z)(xr), 'k--', alpha=0.5, label='Tendencia')
axes[1].set_xlabel('Costo (USD)'); axes[1].set_ylabel('Rating Promedio')
axes[1].set_title('Costo vs Rating por Destino', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=8)

plt.suptitle('Análisis de Correlaciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_correlacion.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Correlaciones con Rating:')
print(corr['rating'].sort_values(ascending=False))

## 12. Conclusiones del EDA

In [ ]:
sparsity_val = 1 - len(df) / (df['user_id'].nunique() * df['destination'].nunique())
ua = df.groupby('user_id').size()

print('=' * 70)
print('CONCLUSIONES DEL EDA — MÓDULO 3 IRNA')
print('=' * 70)
print(f"""
1. CARACTERÍSTICAS GENERALES:
   - {len(df):,} interacciones, {df['user_id'].nunique()} usuarios, {df['destination'].nunique()} destinos.
   - Sparsity {sparsity_val*100:.1f}%: matriz muy dispersa, situación real en recomendación.
   - Ratings aproximadamente normales (mu={df['rating'].mean():.2f}, sigma={df['rating'].std():.2f}).

2. POPULARIDAD:
   - Long-tail clara: pocos destinos concentran la mayoría de reseñas.
   - Top 3: {', '.join(df['destination'].value_counts().head(3).index.tolist())}.
   - Colombia domina ({df[df['country']=='Colombia'].shape[0]/len(df)*100:.0f}% de interacciones).

3. USUARIOS:
   - Promedio {ua.mean():.1f} interacciones/usuario; alta variabilidad (CV={ua.std()/ua.mean():.2f}).
   - {(ua>=3).mean()*100:.0f}% de usuarios con >=3 interacciones: aptos para NCF.

4. CATEGORÍAS:
   - Playa y Ciudad dominan en volumen.
   - Aventura y Ecoturismo: nicho con ratings similares al promedio.

5. IMPLICACIONES PARA NCF:
   - Alta sparsity => embeddings densos son clave para capturar similitudes latentes.
   - Se usará umbral 3.5 para señal implícita binaria (like/dislike).
   - Negative sampling necesario para equilibrar clases.
   - El baseline popular es fuerte por el sesgo de popularidad: el NCF debe superarlo.
""")

# Guardar dataset para notebook 2
df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_viajes.csv'), index=False)
print('Dataset guardado: dataset_viajes.csv')